In [2]:
%load_ext juliacall

In [ ]:

# Latest attempt to make a Matrix object { Julia is integrated into Python using juliacall } 

# Functions like ref,rref,inverse,det and solveLSE are present in the phase-3 of the project, and they will be bridged behind the scenes using juliacall to ensure speed and efficiency.


from juliacall import Main as jl

jl.seval("""

    function matmult( x:: Matrix{<: Number} , y:: Matrix{<: Number})   
        x_dim =  size(x)
        y_dim =  size(y)
        if x_dim[2] != y_dim[1] 
            throw(ArgumentError("Invalid input detectet/dimensions not matching"))
        else 
            # xy = Matrix{Number}(undef,x_dim[1],y_dim[2])
             xy = zeros(Number,x_dim[1],y_dim[2])
            @inbounds @simd for i in 1:1:y_dim[2] 
           @inbounds @simd     for j in 1:1:x_dim[2] 
               @inbounds @simd   for k in 1:1:x_dim[1] 
                        xy[k,i] += x[k,j]*y[j,i]
                    
                                    end
                                end
                            end
            return xy
        end
    end
   """    
        )

jl.seval("""

function add(x:: Matrix{<:Number},y:: Matrix{<: Number})
    x_dim = size(x)
    y_dim = size(y)
    if x_dim != y_dim
        throw(ArgumentError("Dimension Mismatch"))
    else 
        xy = zeros(Number,x_dim[1],x_dim[2])
      @inbounds @simd  for i = 1:1:x_dim[2] # i is  column pointer
         @inbounds @simd   for j = 1:1:x_dim[1] # j is row pointer
                xy[j,i] = x[j,i]+y[j,i]
            end
        end       
    end
    return xy
end




""")

jl.seval("""

function sub(x:: Matrix{<:Number},y:: Matrix{<: Number})
    x_dim = size(x)
    y_dim = size(y)
    if x_dim != y_dim
        throw(ArgumentError("Dimension Mismatch"))
    else 
        xy = zeros(Number,x_dim[1],x_dim[2])
      @inbounds @simd  for i = 1:1:x_dim[2] # i is  column pointer
         @inbounds @simd   for j = 1:1:x_dim[1] # j is row pointer
                xy[j,i] = x[j,i] - y[j,i]
            end
        end       
    end
    return xy
end




""")

jl.seval("""
function transpose_(x::Matrix{<:Number})
    dims = size(x)
    xT =zeros(Number, dims[2],dims[1])
    @inbounds @simd for i = 1:1:dims[1] 
       @inbounds @simd for j in 1:1:dims[2]  
                              xT[j,i] = x[i,j]
        end
    end
return xT
end

function trace(M_raw ::Matrix{<:Number})
    M= M_raw
    sz = size(M)

    if sz[2] != sz[1]
        return throw(ArgumentError("Non-Square Matrices dont have Trace"))
    else
        trace = 0
        for i in 1:sz[1]
            trace += M[i, i]
        end
    end
    return trace
end

"""
        )


class mat ():
    def __init__(self,*rv): # rv = row vectors
        self.__rv = jl.convert(jl.Matrix[jl.Number],jl.transpose(jl.hcat(*rv)))
        
    def iter(self):
        return self.__rv  
        
    def __repr__(self):
        # Leveraging Julia's own function 
        return jl.sprint(jl.show, jl.MIME("text/plain"), self.__rv)
        
    def dim(self) :
        return jl.size(self.iter())
        
    def transpose(self):
       return jl.transpose_(self.iter())
    
    def __mul__(u,v): 
        return jl.matmult(u.iter(),v.iter())

    ''' in julia matrix multiplication
    first column is build first ( using the 3rd loop { k is changing fastest})
    then using 2nd loo the column is updated again and again ( cuz j changes)
    then i changes and we move to the next column ( then above 2 steps are repeated)'''
    def __add__(u,v):
        return jl.add(u.iter(),v.iter())
    def __sub__(u,v):
        return jl.sub(u.iter(),v.iter())
    def trace(u):
        return jl.trace(u.iter())
    def ref(u):
        return jl.ref(u.iter())
    def rref(u):
        return jl.rref(u.iter())
    def solveLSE(u):
        return jl.solveLSE(u.iter())
    def inverse(u):
        return jl.inverse(u.iter())
    def det(u):
        return jl.det(u.iter())
        
            
        
    
        

_IncompleteInputError: incomplete input (1751393276.py, line 148)

In [6]:

# second latest attempt to make a Matrix object

class mat ():
    def __init__(self,*rv): # rv = row vectors
        self.__rv =list(rv)
    def iter(self):
        return self.__rv
    def get_mat(self):
                [print(self.__rv[i]) for i in range ( 0,len(self.__rv),1)]
    def __repr__(self):
        s = ""
        for row in self.__rv:
            s += str(row) + "\n"
        return s.rstrip()
    def rows(self):
        return len(self.iter())
    def columns(self):
        return len(self.iter()[0])
    def dim(self) :
         a = self.rows()
         b = self.columns()
         tup = (a,b)
         return tup
    def transpose(self):
        raw_list = [
            [
                self.iter()[j][i] for j in  range(0,self.rows(),1)
                  ]
            for i in range(0,self.columns(),1)
        ]
        
        return mat(*raw_list)

        
    def __mul__(u,v): 
        u_columns = u.columns()
        v_rows = v.rows()
        if u_columns != v_rows :
            raise TypeError ("dimension mismatch")
        else: 
            u_rows = u.rows()
            v_columns = v.columns()
            u_iter = u.iter()
            v_iter = v.iter()
            product_list = [
                [
                    sum([
                        u_iter[i][k]*v_iter[k][j] for k in range(0,u_columns,1)
                    ]) 
                    for j in range(0,v_columns,1)
                ]
                for i in range(0,u_rows,1)
            ] 
            
            return mat(*product_list)
    def __add__(u,v):
        u_rows = u.rows()
        u_columns = u.columns()
        v_rows = v.rows()
        v_columns = v.columns()
        if u_rows != v_rows or u_columns != v_columns :
            raise TypeError("dimension mismatch")
        else :
            u_iter = u.iter()                       
            v_iter = v.iter()
            
            sum_mat = [
                [
                    u_iter[i][j]+v_iter[i][j] for j in range(0,v_columns,1)
                ]for i in range(0,u_rows,1)
            ]
            return mat(*sum_mat)
    def __sub__(u,v):
        u_rows = u.rows()
        u_columns = u.columns()
        v_rows = v.rows()
        v_columns = v.columns()
        if u_rows != v_rows or u_columns != v_columns :
            raise TypeError("dimension mismatch")
        else :
            u_iter = u.iter()                       
            v_iter = v.iter()
            
            sum_mat = [
                [
                    u_iter[i][j]-v_iter[i][j] for j in range(0,v_columns,1)
                ]for i in range(0,u_rows,1)
            ]
            return mat(*sum_mat)

      

A = mat((2,1),
        (3,9),)
        
B = mat((1,2),
        (3,4))
D = mat([3,10],
        [23,9],
        [12,33])

C = A+B
print(C)
print("---------------------")
A = mat([2,1],
        [3,9],)
        
B = mat([1,2],
        [3,4])

print(A+B)
print("----------------------")
print(B)
print("---------------------------")
print(B.dim())
print("-----------------------")
print(D)

[3, 3]
[6, 13]
---------------------
[3, 3]
[6, 13]
----------------------
[1, 2]
[3, 4]
---------------------------
(2, 2)
-----------------------
[3, 10]
[23, 9]
[12, 33]


In [5]:
# Third attempt to make the matrix object

class mat ():
    def __init__(self,*rv): # rv = row vectors
        self.__rv =rv
    def iter(self):
        return self.__rv
    def get_mat(self):
         for i in range ( 0,len(self.__rv),1):
                print(self.__rv[i])
    def rows(self):
        return len(self.iter())
    def columns(self):
        return len(self.iter()[0])
    def dim(self) :
         a = self.rows()
         b = self.columns()
         tup = (a,b)
         return tup
    def transpose(self):
        raw_tuple = tuple([
            tuple([
                self.iter()[j][i] for j in  range(0,self.rows(),1)
                  ])
            for i in range(0,self.columns(),1)
        ])
        
        return mat(*raw_tuple)

        
    def __mul__(u,v): 
        u_columns = u.columns()
        v_rows = v.rows()
        if u_columns != v_rows :
            raise TypeError ("dimension mismatch")
        else: 
            u_rows = u.rows()
            v_columns = v.columns()
            u_iter = u.iter()
            v_iter = v.iter()
            product_tuple = tuple([
                tuple([
                    sum([
                        u_iter[i][k]*v_iter[k][j] for k in range(0,u_columns,1)
                    ]) 
                    for j in range(0,v_columns,1)
                ])
                for i in range(0,u_rows,1)
            ]) 
            
            return mat(*product_tuple)
    def __add__(u,v):
        u_rows = u.rows()
        u_columns = u.columns()
        v_rows = v.rows()
        v_columns = v.columns()
        if u_rows != v_rows or u_columns != v_columns :
            raise TypeError("dimension mismatch")
        else :
            u_iter = u.iter()                       
            v_iter = v.iter()
            
            sum_mat = tuple([
                tuple([
                    u_iter[i][j]+v_iter[i][j] for j in range(0,v_columns,1)
                ])for i in range(0,u_rows,1)
            ])
            return mat(*sum_mat)
    def __sub__(u,v):
        u_rows = u.rows()
        u_columns = u.columns()
        v_rows = v.rows()
        v_columns = v.columns()
        if u_rows != v_rows or u_columns != v_columns :
            raise TypeError("dimension mismatch")
        else :
            u_iter = u.iter()                       
            v_iter = v.iter()
            
            sum_mat = tuple([
                tuple([
                    u_iter[i][j]-v_iter[i][j] for j in range(0,v_columns,1)
                ])for i in range(0,u_rows,1)
            ])
            return mat(*sum_mat)

      

A = mat((2,1),
        (3,9),
        (4,16))
B = mat((1,2),
        (3,4))
C = mat((1,2),
        (3,4),
        (4,5))
(A-C).get_mat() 
print("--------------")
a = mat((1,2,3))
b = mat((4,5,6))
(a+b).get_mat()

(1, -1)
(0, 5)
(0, 11)
--------------
(5, 7, 9)


In [2]:
# Second attempt to make the matrix class

class mat ():
    def __init__(self,*rv): # rv = row vectors
        self.__rv =rv
    def iter(self):
        return self.__rv
    def get_mat(self):
         for i in range ( 0,len(self.__rv),1):
                print(self.__rv[i])
    def rows(self):
        return len(self.iter())
    def columns(self):
        return len(self.iter()[0])
    def dim(self) :
         a = self.rows()
         b = self.columns()
         tup = (a,b)
         return tup
    def transpose(self):
        raw_tuple= tuple() # empty tuple will act as the basis(skeleton) of the final transposed matrix
        for i in range (0,self.columns(),1):
            temp_tup = tuple() # this temporary tuple will act as the rows of the final transposed matrix
            for j in range(0,self.rows(),1):
                temp_tup += (self.iter()[j][i],) # we are accumulating the column elements of self matrix(by changing j)while i is kept constant{this is similar to double integration or double summation}
                                                 # when one column( of self matrix) is accumulated then it injected in the temp_tup( now its the row of the transposed matrix)
            raw_tuple+= (temp_tup,) # now this temp_tup( which is acting as the row of the transposed matrix) is injected in the raw_tuple
        return mat(*raw_tuple) 
        # lastly this raw_tup is converted into a matrix but with a splat(also known as arg) "*" operator because:
        ''' 1.- so arg when used on an iterator => it opens the iterator element wise and gives it to the function argument
            2.- arg on some variable => packages the incoming parameters as  elements of a tuple( name of tuple will be the variable name)
            3.- since the *cv (in init function) will make it a tuple hence I used the arg on raw_tuple (like *raw_tuple) so they cancel each other out( making the net effect Zero)'''
            
    def mat_mult(u,v): 
        u_columns = u.columns()
        v_rows = v.rows()
        if u_columns != v_rows :
            raise TypeError ("dimension mismatch")
        else: 
            u_rows = u.rows()
            v_columns = v.columns()
            u_iter = u.iter()
            v_iter = v.iter()
            product_tuple = tuple()
            for i in range(0,u_rows,1):
                temp_tuple = tuple()
                for j in range(0,v_columns,1):
                    a =0
                    for k in range(0,u_columns,1):
                        a+= u_iter[i][k]*v_iter[k][j]
                    temp_tuple += (a,)
                product_tuple += (temp_tuple,)
            return mat(*product_tuple) 
                           # same logic as before:
            ''' 1.- so arg when used on an iterator => it opens the iterator element wise and gives it to the function argument
            2.- arg on some variable => packages the incoming parameters as  elements of a tuple( name of tuple will be the variable name)
            3.- since the *cv (in init function) will make it a tuple hence I used the arg on raw_tuple (like *raw_tuple) so they cancel each other out( making the net effect Zero)'''
            
                    


a = mat((1,2))
b = mat((3,4))
b.mat_mult(a.transpose()).get_mat()
         

(11,)


In [1]:
# Earliest attempt to make Matrix class
class vec() :
    def __init__(self,*comp) :
        self.__comp = list(comp)
    def len(self): 
        return len(self.__comp)
    def get_vec(self):
        return self.__comp
    def add(p1,p2) :
        if vec.len(p1) != vec.len(p2) : 
           raise TypeError("dimension mismatch")
            
        else :
            p_net = vec()
            for i in range(0,vec.len(p1),1):
                p_net.get_vec().append(p1.get_vec()[i]+p2.get_vec()[i])
            return p_net
    def add_all(v1,*arg):
        for i in range(0,len(arg),1):
            if v1.len() != arg[i].len() :
                raise TypeError("dimensionally not equal")
            else :
                for j in range(0,arg[i].len(),1):
                    v1.get_vec()[j] = v1.get_vec()[j] + arg[i].get_vec()[j]
        return v1

    def scale(v,a = 1) :
        dummyvec = vec()
        for i in range(0,v.len(),1):
            dummyvec.get_vec().append(v.get_vec()[i]*a)
        return dummyvec
    def mag(v) :
        a = 0
        for i in range(0,v.len(),1):
            a = a+ v.get_vec()[i]**2
        return a**(0.5)
    
    def dot(u,v) :
        a = 0 
        if u.len() != v.len() :
            raise TypeError("dimension mismatch")
        else :
            for i in range(0,u.len(),1):
                a = a +u.get_vec()[i]*v.get_vec()[i]
            return a
    def cosine(u,v):
        w=  vec.dot(u,v)/(u.mag()*v.mag())
        return   w
    def normalise(self) :
        w = self.mag()
        z = self.scale(w)
        return z.get_vec()
    


class mat(vec):
     def __init__(self,*item):
          super().__init__(list(item))
     def get_mat(u):
         return u.get_vec()[0]
     def columns(u) :
         return len(u.get_mat())
     def rows(u) :
         return len(u.get_mat()[0])
     def dot(u,v) :
        a = 0 
        
        for i in range(0,len(u),1):
            a = a +u[i]*v[i]
        return a
     def transpose(u):
         for i in range(0,u.rows(),1):
             for j in range(i+1,u.columns(),1): # only swap till half the matrix, because if you will swap till the last element then two swaps will be executed and hence the net change will be ZERO
                 (u.get_mat()[i][j],u.get_mat()[j][i]) = (u.get_mat()[j][i],u.get_mat()[i][j])
         return u

         "NOT WORKING FROM BELOW THIS, HENCE CODE DESIGN WAS RESTRUCTURED"
         
     """def mat_mult(u,v):
         if u.columns() != v.rows() :
             raise DimensionError("Columns are not equal to Rows")
         else: 
             A = u.rows()
             B = v.columns()
             uT = u.transpose()
             product_list = list()
             for i in range(0,A,1):
                 temp_list = list()
                 for j in range(0,B,1):
                     temp_list.extend(mat.dot(uT.get_mat()[i],v.get_mat()[j]))
                 product_list.extend(temp_list)
             return mat(product_list)"""
                 